# NTO AI Final — Strong Single-Notebook Solution

Гибрид: сильный NoML candidate generation + CatBoostRanker.

In [ ]:
import os
import gc
import math
import random
import warnings
import importlib.util
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from catboost import CatBoostRanker, Pool

warnings.filterwarnings("ignore")
np.random.seed(42)
random.seed(42)


In [ ]:
DATA_DIR_CANDIDATES = [
    '/kaggle/input/datasets/artemnazemtsev/nto-ai',
    './data',
]
TOP_K = 20
CANDS_PER_USER = 500
NEG_PER_USER = 220
HOLDOUT_DAYS = 28
INCIDENT_DAYS = 31

CATBOOST_PARAMS = {
    'loss_function': 'YetiRank',
    'eval_metric': 'NDCG:top=20',
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 8.0,
    'min_data_in_leaf': 30,
    'iterations': 1200,
    'random_seed': 42,
    'verbose': 200,
}

for p in DATA_DIR_CANDIDATES:
    if os.path.exists(p):
        DATA_DIR = p
        break

print('DATA_DIR =', DATA_DIR)


In [ ]:
interactions = pd.read_csv(f'{DATA_DIR}/interactions.csv', parse_dates=['event_ts'], dtype={'user_id':'int64','edition_id':'int64','event_type':'int8'})
targets = pd.read_csv(f'{DATA_DIR}/targets.csv', dtype={'user_id':'int64'})
editions = pd.read_csv(f'{DATA_DIR}/editions.csv')
book_genres = pd.read_csv(f'{DATA_DIR}/book_genres.csv')
users = pd.read_csv(f'{DATA_DIR}/users.csv')

for c in ['edition_id','book_id','author_id','language_id','publisher_id']:
    if c in editions.columns:
        editions[c] = editions[c].fillna(-1).astype('int64')
book_genres['book_id'] = book_genres['book_id'].astype('int64')
book_genres['genre_id'] = book_genres['genre_id'].astype('int64')

pos = interactions[interactions['event_type'].isin([1,2])].copy()
max_ts = pos['event_ts'].max()
incident_start = max_ts - pd.Timedelta(days=INCIDENT_DAYS)
valid_start = incident_start - pd.Timedelta(days=HOLDOUT_DAYS)
print(interactions.shape, targets.shape, editions.shape)
print(valid_start, incident_start, max_ts)


In [ ]:
ed2book = dict(zip(editions['edition_id'], editions['book_id']))
book_to_genres = defaultdict(list)
for b, g in book_genres[['book_id','genre_id']].drop_duplicates().itertuples(index=False):
    book_to_genres[int(b)].append(int(g))
user_pos_full = pos.groupby('user_id')['edition_id'].agg(list).to_dict()


In [ ]:
def time_decay_popularity(df, decay_days=14.0):
    tmax = df['event_ts'].max()
    age = (tmax - df['event_ts']).dt.total_seconds() / 86400.0
    w = np.exp(-age / decay_days)
    out = df.assign(w=w).groupby('edition_id', as_index=False)['w'].sum()
    return out.sort_values(['w','edition_id'], ascending=[False,True]).reset_index(drop=True)

def topk_dict_from_group(frame, key_col, item_col, score_col, k):
    d = {}
    for key, g in frame.groupby(key_col):
        gg = g.sort_values([score_col, item_col], ascending=[False,True]).head(k)
        d[int(key)] = list(zip(gg[item_col].astype(int).tolist(), gg[score_col].astype(float).tolist()))
    return d


In [ ]:
global_pop = pos.groupby('edition_id', as_index=False)['user_id'].nunique().rename(columns={'user_id':'s'})
global_pop = global_pop.sort_values(['s','edition_id'], ascending=[False,True])
decay_pop = time_decay_popularity(pos, decay_days=12.0).rename(columns={'w':'s'})

pos_meta = pos[['user_id','edition_id']].merge(editions[['edition_id','author_id','language_id','publisher_id','book_id']], on='edition_id', how='left')
u_author = pos_meta.groupby(['user_id','author_id'], as_index=False).size().rename(columns={'size':'cnt'})
u_lang = pos_meta.groupby(['user_id','language_id'], as_index=False).size().rename(columns={'size':'cnt'})
u_pub = pos_meta.groupby(['user_id','publisher_id'], as_index=False).size().rename(columns={'size':'cnt'})
for df in [u_author, u_lang, u_pub]:
    df['w'] = df['cnt'] / df.groupby('user_id')['cnt'].transform('sum')

rows = []
for uid, ed, b in pos_meta[['user_id','edition_id','book_id']].itertuples(index=False):
    for g in book_to_genres.get(int(b), []):
        rows.append((int(uid), int(g)))
ugen = pd.DataFrame(rows, columns=['user_id','genre_id'])
u_genre = ugen.groupby(['user_id','genre_id'], as_index=False).size().rename(columns={'size':'cnt'})
u_genre['w'] = u_genre['cnt'] / u_genre.groupby('user_id')['cnt'].transform('sum')


In [ ]:
user_hist = pos.sort_values('event_ts').groupby('user_id')['edition_id'].agg(list)
co_counts = defaultdict(Counter)
for uid, items in tqdm(user_hist.items(), total=len(user_hist), desc='build_co_vis'):
    uniq = []
    s = set()
    for x in items[-80:]:
        x = int(x)
        if x not in s:
            s.add(x)
            uniq.append(x)
    n = len(uniq)
    for i in range(n):
        a = uniq[i]
        for j in range(max(0,i-12), min(n,i+13)):
            if i==j:
                continue
            b = uniq[j]
            co_counts[a][b] += 1

item_pop = pos.groupby('edition_id')['user_id'].nunique().to_dict()
item2top = {}
for a, ctr in tqdm(co_counts.items(), desc='normalize_co_vis'):
    arr = []
    pa = item_pop.get(a,1)
    for b,c in ctr.items():
        pb = item_pop.get(b,1)
        arr.append((b, c / math.sqrt(pa * pb)))
    arr.sort(key=lambda x: (-x[1], x[0]))
    item2top[a] = arr[:120]


In [ ]:
HAS_IMPLICIT = importlib.util.find_spec('implicit') is not None
als_scores = {}
if HAS_IMPLICIT:
    import implicit
    ptmp = pos[['user_id','edition_id','event_type']].copy()
    ptmp['weight'] = np.where(ptmp['event_type']==2, 1.6, 1.1).astype('float32')
    uids = ptmp['user_id'].drop_duplicates().sort_values().to_numpy()
    iids = ptmp['edition_id'].drop_duplicates().sort_values().to_numpy()
    u2i = {u:i for i,u in enumerate(uids)}
    it2i = {it:i for i,it in enumerate(iids)}
    i2it = {i:it for it,i in it2i.items()}
    from scipy.sparse import csr_matrix
    rows = ptmp['user_id'].map(u2i).to_numpy()
    cols = ptmp['edition_id'].map(it2i).to_numpy()
    vals = ptmp['weight'].to_numpy()
    mat = csr_matrix((vals,(rows,cols)), shape=(len(uids),len(iids)))
    als = implicit.als.AlternatingLeastSquares(factors=96, regularization=0.03, iterations=28, random_state=42, use_gpu=False)
    als.fit(mat)
    for u in tqdm(targets['user_id'].tolist(), desc='als_recommend'):
        if u not in u2i:
            continue
        uid = u2i[u]
        recs, scores = als.recommend(uid, mat[uid], N=160, filter_already_liked_items=False)
        als_scores[int(u)] = [(int(i2it[int(ii)]), float(sc)) for ii, sc in zip(recs, scores)]
print('HAS_IMPLICIT=', HAS_IMPLICIT, 'als_users=', len(als_scores))


In [ ]:
global_top = list(zip(global_pop['edition_id'].astype(int).tolist(), global_pop['s'].astype(float).tolist()))
decay_top = list(zip(decay_pop['edition_id'].astype(int).tolist(), decay_pop['s'].astype(float).tolist()))
author2eds = topk_dict_from_group(editions[['edition_id','author_id']].merge(global_pop[['edition_id','s']], on='edition_id', how='left').fillna({'s':0.0}), 'author_id','edition_id','s',120)
lang2eds = topk_dict_from_group(editions[['edition_id','language_id']].merge(global_pop[['edition_id','s']], on='edition_id', how='left').fillna({'s':0.0}), 'language_id','edition_id','s',120)
pub2eds = topk_dict_from_group(editions[['edition_id','publisher_id']].merge(global_pop[['edition_id','s']], on='edition_id', how='left').fillna({'s':0.0}), 'publisher_id','edition_id','s',120)
gtmp = editions[['edition_id','book_id']].merge(book_genres, on='book_id', how='left').merge(global_pop[['edition_id','s']], on='edition_id', how='left').fillna({'s':0.0})
genre2eds = topk_dict_from_group(gtmp, 'genre_id','edition_id','s',120)

ua = u_author.groupby('user_id').apply(lambda g: list(zip(g['author_id'].astype(int), g['w'].astype(float)))).to_dict()
ug = u_genre.groupby('user_id').apply(lambda g: list(zip(g['genre_id'].astype(int), g['w'].astype(float)))).to_dict()
ul = u_lang.groupby('user_id').apply(lambda g: list(zip(g['language_id'].astype(int), g['w'].astype(float)))).to_dict()
up = u_pub.groupby('user_id').apply(lambda g: list(zip(g['publisher_id'].astype(int), g['w'].astype(float)))).to_dict()

def build_user_candidates(uid, max_items=CANDS_PER_USER):
    uid = int(uid)
    seen = set(user_pos_full.get(uid, []))
    scores = defaultdict(float)
    src_mask = defaultdict(int)
    for rank, (ed, s) in enumerate(decay_top[:220], start=1):
        if ed in seen: continue
        scores[ed] += 1.00 * (s / (1.0 + rank*0.01)); src_mask[ed] |= 1
    for rank, (ed, s) in enumerate(global_top[:240], start=1):
        if ed in seen: continue
        scores[ed] += 0.55 * (s / (1.0 + rank*0.01)); src_mask[ed] |= 2
    for idx, item in enumerate(user_pos_full.get(uid, [])[-30:]):
        mult = 0.8 + (0.6 * (idx+1) / max(1, len(user_pos_full.get(uid, [])[-30:])))
        for nb, s in item2top.get(int(item), []):
            if nb in seen: continue
            scores[nb] += 1.35 * mult * s; src_mask[nb] |= 4
    for aid, w in ua.get(uid, [])[:14]:
        for ed, s in author2eds.get(aid, [])[:60]:
            if ed in seen: continue
            scores[ed] += 1.15 * w * (1 + math.log1p(s)); src_mask[ed] |= 8
    for gid, w in ug.get(uid, [])[:14]:
        for ed, s in genre2eds.get(gid, [])[:60]:
            if ed in seen: continue
            scores[ed] += 1.10 * w * (1 + math.log1p(s)); src_mask[ed] |= 16
    for lid, w in ul.get(uid, [])[:8]:
        for ed, s in lang2eds.get(lid, [])[:70]:
            if ed in seen: continue
            scores[ed] += 0.90 * w * (1 + math.log1p(s)); src_mask[ed] |= 32
    for pid, w in up.get(uid, [])[:8]:
        for ed, s in pub2eds.get(pid, [])[:70]:
            if ed in seen: continue
            scores[ed] += 0.80 * w * (1 + math.log1p(s)); src_mask[ed] |= 64
    for ed, s in als_scores.get(uid, [])[:150]:
        if ed in seen: continue
        scores[ed] += 1.25 * max(0.0, s); src_mask[ed] |= 128
    ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:max_items]
    return ranked, src_mask


In [ ]:
cand_rows = []
for uid in tqdm(targets['user_id'].tolist(), desc='build_candidates'):
    ranked, src_mask = build_user_candidates(uid, max_items=CANDS_PER_USER)
    for ed, s in ranked:
        cand_rows.append((int(uid), int(ed), float(s), int(src_mask[ed])))
cands = pd.DataFrame(cand_rows, columns=['user_id','edition_id','cand_score','src_mask'])
print(cands.shape, cands.shape[0] / max(1, targets.shape[0]))


In [ ]:
pop_all = pos.groupby('edition_id').size().rename('pop_all')
pop_recent = pos[pos['event_ts'] >= incident_start].groupby('edition_id').size().rename('pop_recent')
uniq_users = pos.groupby('edition_id')['user_id'].nunique().rename('item_uniq_users')
item_feat = pd.concat([pop_all, pop_recent, uniq_users], axis=1).fillna(0).reset_index()
item_feat['pop_all_log'] = np.log1p(item_feat['pop_all'])
item_feat['pop_recent_log'] = np.log1p(item_feat['pop_recent'])

u_all = pos.groupby('user_id').size().rename('u_all')
u_recent = pos[pos['event_ts'] >= incident_start].groupby('user_id').size().rename('u_recent')
u_uniq = pos.groupby('user_id')['edition_id'].nunique().rename('u_uniq_ed')
user_feat = pd.concat([u_all, u_recent, u_uniq], axis=1).fillna(0).reset_index()
user_feat['u_all_log'] = np.log1p(user_feat['u_all'])
user_feat['u_recent_ratio'] = user_feat['u_recent'] / (1 + user_feat['u_all'])

user_author_w = u_author.set_index(['user_id','author_id'])['w']
user_lang_w = u_lang.set_index(['user_id','language_id'])['w']
user_pub_w = u_pub.set_index(['user_id','publisher_id'])['w']
user_genre_w = u_genre.set_index(['user_id','genre_id'])['w']

def genre_aff(uid, book_id):
    gs = book_to_genres.get(int(book_id), [])
    best = 0.0
    for g in gs:
        v = float(user_genre_w.get((int(uid), int(g)), 0.0))
        if v > best: best = v
    return best

cands = cands.merge(editions[['edition_id','author_id','language_id','publisher_id','book_id']], on='edition_id', how='left')
cands['u_author_w'] = [float(user_author_w.get((u,a), 0.0)) for u,a in cands[['user_id','author_id']].itertuples(index=False)]
cands['u_lang_w'] = [float(user_lang_w.get((u,l), 0.0)) for u,l in cands[['user_id','language_id']].itertuples(index=False)]
cands['u_pub_w'] = [float(user_pub_w.get((u,p), 0.0)) for u,p in cands[['user_id','publisher_id']].itertuples(index=False)]
cands['u_genre_w'] = [genre_aff(u, b) for u,b in cands[['user_id','book_id']].itertuples(index=False)]
cands = cands.merge(item_feat, on='edition_id', how='left').merge(user_feat, on='user_id', how='left').fillna(0)
for bit, name in [(1,'src_decay'),(2,'src_global'),(4,'src_covisit'),(8,'src_author'),(16,'src_genre'),(32,'src_lang'),(64,'src_pub'),(128,'src_als')]:
    cands[name] = ((cands['src_mask'] & bit) > 0).astype('int8')
cands['cand_score_log'] = np.log1p(np.maximum(cands['cand_score'], 0))


In [ ]:
valid_truth = pos[(pos['event_ts'] >= valid_start) & (pos['event_ts'] < incident_start)][['user_id','edition_id']].drop_duplicates().copy()
train_users = valid_truth['user_id'].drop_duplicates().tolist()
train_rows = []
for uid in tqdm(train_users, desc='build_train_rows'):
    ranked, src_mask = build_user_candidates(uid, max_items=CANDS_PER_USER)
    pos_items = set(valid_truth.loc[valid_truth['user_id']==uid, 'edition_id'].tolist())
    pos_cnt = 0
    neg_cnt = 0
    for ed, s in ranked:
        is_pos = int(ed in pos_items)
        if is_pos and pos_cnt < 80:
            train_rows.append((uid, ed, s, src_mask[ed], 1)); pos_cnt += 1
        elif (not is_pos) and neg_cnt < NEG_PER_USER:
            train_rows.append((uid, ed, s, src_mask[ed], 0)); neg_cnt += 1
        if neg_cnt >= NEG_PER_USER and pos_cnt >= len(pos_items):
            break
train_df = pd.DataFrame(train_rows, columns=['user_id','edition_id','cand_score','src_mask','label'])
print(train_df.shape, train_df['label'].mean())


In [ ]:
train_df = train_df.merge(editions[['edition_id','author_id','language_id','publisher_id','book_id']], on='edition_id', how='left')
train_df['u_author_w'] = [float(user_author_w.get((u,a), 0.0)) for u,a in train_df[['user_id','author_id']].itertuples(index=False)]
train_df['u_lang_w'] = [float(user_lang_w.get((u,l), 0.0)) for u,l in train_df[['user_id','language_id']].itertuples(index=False)]
train_df['u_pub_w'] = [float(user_pub_w.get((u,p), 0.0)) for u,p in train_df[['user_id','publisher_id']].itertuples(index=False)]
train_df['u_genre_w'] = [genre_aff(u, b) for u,b in train_df[['user_id','book_id']].itertuples(index=False)]
train_df = train_df.merge(item_feat, on='edition_id', how='left').merge(user_feat, on='user_id', how='left').fillna(0)
for bit, name in [(1,'src_decay'),(2,'src_global'),(4,'src_covisit'),(8,'src_author'),(16,'src_genre'),(32,'src_lang'),(64,'src_pub'),(128,'src_als')]:
    train_df[name] = ((train_df['src_mask'] & bit) > 0).astype('int8')
train_df['cand_score_log'] = np.log1p(np.maximum(train_df['cand_score'], 0))

feature_cols = [
    'cand_score','cand_score_log','u_author_w','u_lang_w','u_pub_w','u_genre_w',
    'pop_all','pop_recent','item_uniq_users','pop_all_log','pop_recent_log',
    'u_all','u_recent','u_uniq_ed','u_all_log','u_recent_ratio',
    'src_decay','src_global','src_covisit','src_author','src_genre','src_lang','src_pub','src_als',
    'author_id','language_id','publisher_id'
]
cat_features = [feature_cols.index(c) for c in ['author_id','language_id','publisher_id']]


In [ ]:
train_df = train_df.sort_values(['user_id']).reset_index(drop=True)
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df['label'].astype('float32').to_numpy(),
    group_id=train_df['user_id'].to_numpy(),
    cat_features=cat_features,
)
model = CatBoostRanker(**CATBOOST_PARAMS)
model.fit(train_pool)


In [ ]:
cands_pred = cands.sort_values(['user_id']).reset_index(drop=True)
pred_pool = Pool(data=cands_pred[feature_cols], group_id=cands_pred['user_id'].to_numpy(), cat_features=cat_features)
cands_pred['pred'] = model.predict(pred_pool)
cands_pred['final'] = cands_pred['pred'] + 0.18 * cands_pred['cand_score_log'] + 0.04 * (
    cands_pred['src_covisit'] + cands_pred['src_author'] + cands_pred['src_genre'] + cands_pred['src_als']
)
cands_pred = cands_pred.sort_values(['user_id','final','edition_id'], ascending=[True,False,True])


In [ ]:
all_global = [int(x) for x in decay_pop['edition_id'].tolist()]
rows = []
for uid, g in tqdm(cands_pred.groupby('user_id'), total=targets['user_id'].nunique(), desc='make_submission'):
    uid = int(uid)
    seen = set(user_pos_full.get(uid, []))
    picked, used = [], set()
    for ed in g['edition_id'].astype(int).tolist():
        if ed in used or ed in seen: continue
        used.add(ed); picked.append(ed)
        if len(picked) == TOP_K: break
    if len(picked) < TOP_K:
        for ed in all_global:
            if ed in used or ed in seen: continue
            used.add(ed); picked.append(ed)
            if len(picked) == TOP_K: break
    for r, ed in enumerate(picked, start=1):
        rows.append((uid, int(ed), int(r)))
submission = pd.DataFrame(rows, columns=['user_id','edition_id','rank'])
print(submission.shape)
print(submission.groupby('user_id').size().min(), submission.groupby('user_id').size().max())
submission.to_csv('submission.csv', index=False)
submission.head(20)
